### **Chapter 9.8: DeePC With Noisy Data - Fit Residual, Direct vs. Indirect, and Tuning**

Chapter 9.5 introduced the regularized DeePC problem

$$
\min_g\; \|Y_f g\|_{\bar Q}^2 + \|U_f g\|_{\bar R}^2 + \lambda_g \|g\|_2^2 + \lambda_y \|Y_{\mathrm{ini}}\, g - y_{\mathrm{meas}}\|_2^2
$$

as a robustification for data that are not exactly consistent with an LTI behavior. This notebook looks at the **linear plant with measurement noise** and asks three questions:

1. What does the fitted output residual $\sigma_y := \|Y_{\mathrm{ini}}\, g^\star - y_{\mathrm{meas}}\|_2$ tell us? It is the part of the measured history that the data behavior *cannot* explain - an online indicator of model mismatch.
2. How close does DeePC get to the MPC that knows the true model, and how does it compare with the classical **indirect** route - identify a model from the *same* noisy data by least squares, then run MPC on it?
3. How sensitive is the result to $\lambda_g, \lambda_y$?

Setup: flat Mountain Car, position-only output $y=p$, Gaussian measurement noise with standard deviation $\sigma$ on the offline data *and* on the online measurements. All controllers use the same cost, horizon, constraints and OSQP backend. The model-based controllers are `ARXPredictiveController` instances: the **oracle** uses the exact ZOH model $p_{k+1}=2p_k-p_{k-1}+\tfrac{\Delta t^2}{2}(u_k+u_{k-1})$; the **indirect** controller uses the ARX(2,2) model fitted by ordinary least squares to the noisy data (`identify_arx_least_squares`).

In [ ]:
import sys
import os
import io
import contextlib
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from utils.env import *
from ex9_DeePC.deepc_utils import *

OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
plt.rcParams.update({
    "font.size": 11, "axes.labelsize": 11, "axes.titlesize": 11, "legend.fontsize": 9,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "figure.dpi": 110, "savefig.dpi": 300,
    "axes.grid": True, "grid.alpha": 0.3, "lines.linewidth": 1.8,
    "axes.prop_cycle": plt.cycler(color=OKABE_ITO),
})
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), bbox_inches="tight")

In [ ]:
freq = 20
dt = 1.0 / freq
N = 20
T_ini = 4
T_data = 400
t_terminal = 6.0

initial_state = np.array([-0.5, 0.0])
target_state = np.array([0.6, 0.0])
env = Env(1, initial_state, target_state, input_lbs=-1.0, input_ubs=1.0)
dynamics = Dynamics(env)

Q_y = np.array([[1.0]])
R = np.array([[0.1]])
Qf_y = Q_y
u_eq = float(dynamics.get_equilibrium_input(target_state))
a_true, b_true = true_arx_double_integrator(dt)

u_clean, y_clean = collect_deepc_data(env, dynamics, freq=freq, n_samples=T_data, excitation_amplitude=0.8,
                                      initial_state=target_state, seed=1, output_indices=[0])

def noisy_data(sigma, seed):
    rng = np.random.default_rng(1000 + seed)
    return u_clean, y_clean + rng.normal(0.0, sigma, y_clean.shape)

def make_oracle():
    return ARXPredictiveController(env, dynamics, a_true, b_true, Q_y, R, Qf_y, freq, N, name="oracle")

def make_indirect(u_d, y_d):
    a_hat, b_hat, _ = identify_arx_least_squares(u_d, y_d, order=2, input_offset=u_eq, output_offset=target_state[0])
    return ARXPredictiveController(env, dynamics, a_hat, b_hat, Q_y, R, Qf_y, freq, N, name="indirect")

def make_deepc(u_d, y_d, lambda_g=1.0, lambda_y=1e4):
    return DeePCController(env, dynamics, u_d, y_d, Q_y, R, Qf_y, freq, N, T_ini=T_ini, lambda_g=lambda_g,
                           lambda_y=lambda_y, output_indices=[0], history_initialization='equilibrium', verbose=False)

def run_closed_loop(controller, sigma_meas, seed, plant=dynamics, env_run=env, t_end=t_terminal):
    # The controller sees position measurements corrupted by N(0, sigma^2); the
    # cost is evaluated on the true state.
    rng = np.random.default_rng(2000 + seed)
    x = env_run.init_state.copy()
    states, inputs, residuals, pred_errors, failures = [x.copy()], [], [], [], 0
    y_next_pred = None
    for k in range(int(freq * t_end)):
        x_meas = x.copy(); x_meas[0] += rng.normal(0.0, sigma_meas)
        if y_next_pred is not None:                      # one-step-ahead prediction error on the measurement
            pred_errors.append(abs(x_meas[0] - y_next_pred))
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                out = controller.compute_action(x_meas, k)
            u, y_pred = (out[0], out[1]) if isinstance(out, tuple) else (out, None)
            y_next_pred = float(np.asarray(y_pred)[1, 0]) if y_pred is not None else None
            if isinstance(controller, DeePCController):
                residuals.append(controller.last_output_residual)
        except RuntimeError:
            failures += 1; u = np.zeros(1)
            if isinstance(controller, DeePCController):
                controller.initialize_history(x_meas, mode='equilibrium')
        u = np.clip(np.asarray(u, dtype=float).reshape(-1), env_run.input_lbs, env_run.input_ubs)
        x = plant.one_step_forward(x, u, dt)
        states.append(x.copy()); inputs.append(u.copy())
    return np.asarray(states), np.asarray(inputs), np.asarray(residuals), failures, np.asarray(pred_errors)

def closed_loop_cost(states, inputs):
    dp = states[:, 0] - target_state[0]
    du = inputs[:, 0] - u_eq
    return float(np.sum(Q_y[0, 0] * dp[:-1]**2 + R[0, 0] * du**2) + Qf_y[0, 0] * dp[-1]**2)

### **Part 1: The Output Residual as a Mismatch Indicator**

With a hard output-history constraint the residual is zero by construction. In the regularized problem the measured history is *fitted*, and

$$
\sigma_y = \|Y_{\mathrm{ini}}\, g^\star - y_{\mathrm{meas}}\|_2
$$

is what the recorded behavior cannot reproduce. Two candidate causes are **measurement noise** (the recorded and the measured trajectories are both perturbed) and **plant mismatch** (the plant that generates $y_{\mathrm{meas}}$ is not the plant that generated the data). We sweep both, record the mean residual over a closed-loop run, and put next to it the quantity one would usually monitor, the **one-step-ahead prediction error** $|y_{k} - \hat y_{k|k-1}|$ on the measurement. Regularization weights are $\lambda_g = 1$, $\lambda_y = 10^4$ throughout (chosen in Example 2.1).

In [ ]:
sigma_values = [0.0, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]
bump_values = [0.0, 5e-4, 1e-3, 2e-3, 3e-3, 4e-3, 5e-3]

res_noise, res_bump = [], []
for sigma in sigma_values:
    u_d, y_d = noisy_data(sigma, seed=0)
    _, _, r, _, e = run_closed_loop(make_deepc(u_d, y_d), sigma, seed=0)
    rank = np.linalg.matrix_rank(np.vstack([make_deepc(u_d, y_d).Up, make_deepc(u_d, y_d).Yp, make_deepc(u_d, y_d).Uf, make_deepc(u_d, y_d).Yf]))
    res_noise.append((np.mean(r), np.mean(e), rank))
for k_bump in bump_values:
    env_b = Env(3, initial_state, target_state, param=k_bump, input_lbs=-1.0, input_ubs=1.0)
    _, _, r, _, e = run_closed_loop(make_deepc(u_clean, y_clean), 0.0, seed=0, plant=Dynamics(env_b), env_run=env_b)
    res_bump.append((np.mean(r), np.mean(e)))

print(f"{'sigma (m)':>10} {'fit residual':>13} {'1-step error':>13} {'rank(H)':>8}      {'bump k':>8} {'fit residual':>13} {'1-step error':>13}")
for (s, (m1, e1, rk)), (k, (m2, e2)) in zip(zip(sigma_values, res_noise), zip(bump_values, res_bump)):
    print(f"{s:10.0e} {m1:13.2e} {e1:13.2e} {rk:8d}      {k:8.4f} {m2:13.2e} {e2:13.2e}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
s_plot = np.array(sigma_values); s_plot[0] = 3e-5     # place sigma = 0 at the left edge of the log axis
ax[0].loglog(s_plot, [r[0] for r in res_noise], marker="o", label=r"fit residual $\sigma_y$")
ax[0].loglog(s_plot, [r[1] for r in res_noise], marker="s", ls="--", label=r"one-step prediction error $|y_k-\hat y_{k|k-1}|$")
ax[0].loglog(s_plot[1:], s_plot[1:], color="0.6", ls=":", label=r"$\sigma$")
ax[0].set_xlabel(r"noise std $\sigma$ on data and measurements (m)   [leftmost point: $\sigma=0$]"); ax[0].set_ylabel("m")
ax[0].set_title("Measurement noise (flat plant)"); ax[0].legend()
ax[1].semilogy(bump_values, [r[0] for r in res_bump], marker="o", label=r"fit residual $\sigma_y$")
ax[1].semilogy(bump_values, [r[1] for r in res_bump], marker="s", ls="--", label="one-step prediction error")
ax[1].set_xlabel("bump amplitude $k$ of the plant (flat data, no noise)"); ax[1].set_ylabel("m")
ax[1].set_title("Plant mismatch"); ax[1].legend()
plt.tight_layout()
savefig(fig, "9.8_residual_indicator")
plt.show()

The two indicators tell different stories, and the difference is instructive:

- **Plant mismatch** (right): with noise-free data the stacked Hankel matrix has rank $mL+n = 26$ and spans exactly the flat behavior. A history generated by the bumpy plant is *not* in that span, so the fit residual grows monotonically with $k$ (1.0 mm at $k=0$, the small floor set by $\lambda_y$, to 2.8 mm at $k = 0.005$), and so does the prediction error. Here $\sigma_y$ is what one hopes for: an online consistency check of the data model.
- **Measurement noise** (left): the residual does **not** grow with $\sigma$ - it even decreases. Noise on the recorded data makes the Hankel matrix *full rank* (49 instead of 26, see the printed table), so the columns can reproduce any history, noisy or not, and the fit becomes *easier*. The residual then only reflects the $\lambda_g/\lambda_y$ trade-off. The one-step prediction error, in contrast, tracks the noise (and its amplification through the poorly determined velocity) faithfully.

So $\sigma_y$ is a measure of *structured* inconsistency between the history and the data - useful for detecting plant changes - but not of data quality; the latter shows up in the prediction error and, as Part 2 shows, in the closed-loop cost. This is the same rank observation that makes regularization indispensable on the nonlinear plant in Chapter 9.6, seen from the other side.

### **Part 2: Direct vs. Indirect Data-Driven Control**

Three controllers, all with the same cost and constraints, judged by the closed-loop cost evaluated on the *true* state:

- **Oracle**: MPC with the exact model - the best any of them can do.
- **Indirect**: least-squares ARX(2,2) fitted to the noisy data, then MPC on the fitted model. With output noise the regressors are noisy as well, so plain least squares is biased (errors in variables); the bias grows with $\sigma$.
- **Direct**: regularized DeePC on the same noisy data with one fixed setting $\lambda_g = 1, \lambda_y = 10^{4}$ (Example 2.1 shows how this was chosen and how much it matters).

Each noise level is repeated for 10 independent noise realizations (data and measurements).

In [ ]:
sigma_grid = [0.0, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
n_seeds = 10
costs = {"oracle": np.zeros((len(sigma_grid), n_seeds)), "indirect": np.zeros((len(sigma_grid), n_seeds)), "DeePC": np.zeros((len(sigma_grid), n_seeds))}
model_errors = np.zeros((len(sigma_grid), n_seeds))

for i, sigma in enumerate(sigma_grid):
    for s in range(n_seeds):
        u_d, y_d = noisy_data(sigma, seed=s)
        a_hat, b_hat, _ = identify_arx_least_squares(u_d, y_d, order=2, input_offset=u_eq, output_offset=target_state[0])
        model_errors[i, s] = np.linalg.norm(np.concatenate([a_hat - a_true, (b_hat - b_true) / b_true]))
        for name, factory in [("oracle", make_oracle), ("indirect", lambda: make_indirect(u_d, y_d)), ("DeePC", lambda: make_deepc(u_d, y_d))]:
            X, U, _, fails, _ = run_closed_loop(factory(), sigma, seed=s)
            costs[name][i, s] = closed_loop_cost(X, U)

print(f"{'sigma (m)':>10} | {'oracle':>16} | {'indirect (LS-ARX + MPC)':>24} | {'DeePC (direct)':>16} | {'LS model error':>14}")
for i, sigma in enumerate(sigma_grid):
    row = " | ".join(f"{costs[n][i].mean():8.3f} +- {costs[n][i].std():6.3f}" for n in ["oracle", "indirect", "DeePC"])
    print(f"{sigma:10.0e} | {row} | {model_errors[i].mean():14.3f}")

In [ ]:
print()
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.8), width_ratios=[1.3, 1])
s_plot = np.array(sigma_grid); s_plot[0] = 3e-5
for j, (name, label) in enumerate([("oracle", "oracle: MPC, true model"), ("indirect", "indirect: LS-ARX + MPC"), ("DeePC", "direct: regularized DeePC")]):
    med = np.median(costs[name], axis=1); lo = np.percentile(costs[name], 10, axis=1); hi = np.percentile(costs[name], 90, axis=1)
    ax[0].plot(s_plot, med, marker="o", label=label, color=OKABE_ITO[j])
    ax[0].fill_between(s_plot, lo, hi, color=OKABE_ITO[j], alpha=0.18)
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel(r"noise std $\sigma$ on data and measurements (m)   [leftmost point: $\sigma=0$]"); ax[0].set_ylabel("closed-loop cost (true state)")
ax[0].set_title("Median and 10-90 % band over 10 noise realizations"); ax[0].legend()

ax[1].semilogx(s_plot, np.median(model_errors, axis=1), marker="o", color=OKABE_ITO[1])
ax[1].fill_between(s_plot, np.percentile(model_errors, 10, axis=1), np.percentile(model_errors, 90, axis=1), color=OKABE_ITO[1], alpha=0.18)
ax[1].set_xlabel(r"noise std $\sigma$ (m)"); ax[1].set_ylabel(r"$\|[\hat a - a,\; (\hat b - b)/b]\|_2$")
ax[1].set_title("Least-squares model error (indirect route)")
plt.tight_layout()
savefig(fig, "9.8_direct_vs_indirect")
plt.show()

In [ ]:
# One realization at the largest noise level, to see what the numbers mean.
sigma_show = sigma_grid[-1]; s = 0
u_d, y_d = noisy_data(sigma_show, seed=s)
runs = {}
for name, factory in [("oracle", make_oracle), ("indirect", lambda: make_indirect(u_d, y_d)), ("DeePC", lambda: make_deepc(u_d, y_d))]:
    runs[name] = run_closed_loop(factory(), sigma_show, seed=s)
a_hat, b_hat, _ = identify_arx_least_squares(u_d, y_d, order=2, input_offset=u_eq, output_offset=target_state[0])
print(f"sigma = {sigma_show}: identified a = {a_hat.round(3)}, b = {b_hat.round(5)}  (true a = {a_true}, b = {b_true.round(5)})")

t_x = np.arange(int(freq * t_terminal) + 1) * dt; t_u = t_x[:-1]
fig, ax = plt.subplots(2, 1, figsize=(9, 5.2), sharex=True)
for j, (name, label) in enumerate([("oracle", "oracle"), ("indirect", "indirect (LS-ARX + MPC)"), ("DeePC", "direct (DeePC)")]):
    X, U, _, _, _ = runs[name]
    ax[0].plot(t_x, X[:, 0], label=f"{label}, cost {closed_loop_cost(X, U):.2f}", color=OKABE_ITO[j])
    ax[1].step(t_u, U[:, 0], where="post", color=OKABE_ITO[j], lw=1.2)
ax[0].axhline(target_state[0], color="0.4", ls=":"); ax[0].set_ylabel("position (true)"); ax[0].legend(); ax[0].set_title(f"One realization at $\\sigma$ = {sigma_show} m")
ax[1].set_ylabel("input"); ax[1].set_xlabel("time (s)")
plt.tight_layout()
savefig(fig, "9.8_trajectories_sigma_max")
plt.show()

- Up to $\sigma = 3\cdot10^{-4}$ m all three are indistinguishable (within 0.5 % of each other); the indirect controller is even marginally better than DeePC, whose soft history fit costs a constant 0.4 %.
- From $\sigma = 10^{-3}$ m the least-squares model degrades (relative parameter error 0.2 at $10^{-3}$, 0.7 at $3\cdot10^{-3}$, 2.2 at $10^{-2}$; at $\sigma=10^{-2}$ the identified $a = [1.1, -0.1]$ instead of $[2, -1]$ no longer describes a double integrator) and so does its controller: $2\times$ the oracle cost at $3\cdot10^{-3}$ m, $4.5\times$ with a huge spread at $10^{-2}$ m. DeePC with fixed $(\lambda_g, \lambda_y)$ stays within 5 % of the oracle up to $3\cdot10^{-3}$ m and at about $2\times$ at $10^{-2}$ m, with a small spread across realizations.
- The reason is *not* that DeePC has no model: with output noise on both regressors and targets, ordinary least squares is biased (errors in variables), and the bias is squared into the closed loop. DeePC never commits to a parametric model; the regularized $g$ fits the noisy history in a least-squares sense at every step. Better identification (instrumental variables, subspace methods, more data) narrows the gap - the point of the comparison is that the *same* data and the *same* effort give a more robust controller when identification and control are not separated.

#### **Example 2.1: Tuning sensitivity of the direct approach**

The direct result above used one hand-picked $(\lambda_g, \lambda_y)$. How wide is the window? We fix $\sigma = 3\cdot10^{-3}$ m, sweep both regularization weights over several decades (3 noise realizations each) and show the median cost relative to the oracle. For orientation: Chapter 9.5 used $\lambda_g = 10^{-3}, \lambda_y = 10^3$ with full-state output.

In [ ]:
sigma_tune = 3e-3
lambda_g_grid = [1e-3, 1e-2, 1e-1, 1.0, 10.0, 1e2, 1e3]
lambda_y_grid = [1e3, 1e4, 1e5, 1e6, 1e7]
n_seeds_tune = 3
J_or = np.mean([closed_loop_cost(*run_closed_loop(make_oracle(), sigma_tune, seed=s)[:2]) for s in range(n_seeds_tune)])
J_ind = np.mean([closed_loop_cost(*run_closed_loop(make_indirect(*noisy_data(sigma_tune, s)), sigma_tune, seed=s)[:2]) for s in range(n_seeds_tune)])
grid = np.zeros((len(lambda_y_grid), len(lambda_g_grid)))
for iy, ly in enumerate(lambda_y_grid):
    for ig, lg in enumerate(lambda_g_grid):
        vals = []
        for s in range(n_seeds_tune):
            u_d, y_d = noisy_data(sigma_tune, s)
            X, U, _, fails, _ = run_closed_loop(make_deepc(u_d, y_d, lambda_g=lg, lambda_y=ly), sigma_tune, seed=s)
            vals.append(closed_loop_cost(X, U))
        grid[iy, ig] = np.median(vals)
print(f"sigma = {sigma_tune}: oracle cost {J_or:.3f}, indirect cost {J_ind:.3f}")
print(f"DeePC cost / oracle cost: best {grid.min()/J_or:.3f} at lambda_g={lambda_g_grid[np.unravel_index(grid.argmin(), grid.shape)[1]]:.0e}, "
      f"lambda_y={lambda_y_grid[np.unravel_index(grid.argmin(), grid.shape)[0]]:.0e}; worst {grid.max()/J_or:.2f}")

from matplotlib.colors import LogNorm
fig, ax = plt.subplots(figsize=(7.2, 3.9))
im = ax.imshow(grid / J_or, origin="lower", cmap="viridis", norm=LogNorm(vmin=1.0, vmax=max(2.0, (grid / J_or).max())), aspect="auto")
ax.set_xticks(range(len(lambda_g_grid))); ax.set_xticklabels([f"$10^{{{int(np.log10(v))}}}$" for v in lambda_g_grid])
ax.set_yticks(range(len(lambda_y_grid))); ax.set_yticklabels([f"$10^{{{int(np.log10(v))}}}$" for v in lambda_y_grid])
ax.set_xlabel(r"$\lambda_g$"); ax.set_ylabel(r"$\lambda_y$"); ax.grid(False)
for iy in range(len(lambda_y_grid)):
    for ig in range(len(lambda_g_grid)):
        v = grid[iy, ig] / J_or
        ax.text(ig, iy, f"{v:.2f}" if v < 100 else f"{v:.0f}", ha="center", va="center", fontsize=7.5, color="w" if v > 1.6 else "k")
cb = plt.colorbar(im, ax=ax); cb.set_label("DeePC cost / oracle cost")
ax.set_title(f"Tuning map at $\\sigma$ = {sigma_tune} m (indirect: {J_ind/J_or:.2f} x oracle)")
plt.tight_layout()
savefig(fig, "9.8_tuning_map")
plt.show()

The map shows why "DeePC is sensitive to tuning" is a fair statement. The good region (within 20 % of the oracle) is $\lambda_g \in [1, 10]$ with $\lambda_y \in [10^3, 10^5]$; one decade less $\lambda_g$ doubles the cost, two decades less give $9$-$17\times$ (the coefficient overfits the noise), and the Chapter 9.5 setting $(10^{-3}, 10^3)$ - fine for full-state output on the flat plant - is $9\times$ the oracle here. Large $\lambda_g$ ($\ge 10^2$) saturates at $2.3$-$2.8\times$: the prediction is biased towards "nothing happens". Large $\lambda_y$ with small $\lambda_g$ is the worst corner ($26\times$): the history is matched exactly with a huge, noise-fitted $g$. Note also that the two weights interact - the optimum shifts with $\sigma$ and with the output dimension - so a tuning map like this, or a validation-based rule, belongs to every DeePC deployment.

<blockquote style="padding: 18px 20px; margin: 1.2em 0; background: rgba(56, 139, 253, 0.12); border-left: 4px solid rgba(56, 139, 253, 0.85); border-radius: 6px; color: inherit !important;">

##### **Takeaway: regularized DeePC is a (well-behaved) indirect method in disguise**

The fitted residual $\sigma_y$ measures the *structured* inconsistency between the measured history and the recorded behavior: it tracks plant mismatch but is blind to noise, because noise makes the Hankel matrix full rank. On the noisy linear plant, regularized DeePC stays within 5 % of the true-model MPC up to millimetre-level noise and degrades gracefully beyond, while the textbook indirect route - least-squares ARX identification followed by MPC - degrades much faster as the identified model absorbs the output noise. The direct approach is not free of modelling choices, however: $\lambda_g$ and $\lambda_y$ have to be chosen, and the good region, while broad, is bounded on both sides (too little regularization overfits the noise, too much biases the prediction). Dörfler, Coulson and Markovsky show that this is no coincidence: regularized DeePC is equivalent to an indirect scheme in which the identification and control steps are solved *jointly* with a relaxed low-rank/least-squares constraint - which is exactly why it can be more robust than solving them one after the other.
</blockquote>

**References:** Dörfler, Coulson, and Markovsky, *Bridging Direct & Indirect Data-Driven Control Formulations via Regularizations and Relaxations*, IEEE TAC 2023 (arXiv:2101.01273); Coulson, Lygeros, and Dörfler, *Regularized and Distributionally Robust Data-Enabled Predictive Control*, CDC 2019; Berberich, Köhler, Müller, and Allgöwer, *Data-Driven Model Predictive Control With Stability and Robustness Guarantees*, IEEE TAC 2021.